In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import logging
import os
from IPython.display import display

from gsm_benchmarker.results_analyser.prompt_result import PromptResult
from gsm_benchmarker.results_analyser.plotting_utils import Colour, plot_prompt_comparison, plot_prompt_acc_evolution

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
METRIC = "correct"

In [ ]:
MARGIN = 0.1

ALPHA = 0.05
ALPHA_THRESHOLD = (1 + MARGIN) * ALPHA
PROJECTED_ALPHA = 0.2
PROJECTED_ALPHA_THRESHOLD = (1 + MARGIN) * PROJECTED_ALPHA

In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
result_kwargs = dict(metric=METRIC, notebook=True, save_dest=OUTPUTS)
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()


pilot_gsm_result = PromptResult(
    pp / "mini_20x50x4__14_11/final",
    colour=Colour('green'),
    full_label="GSM prompt (pilot study)",
    short_label="GSM_pilot",
    **result_kwargs
)

gsm_result = PromptResult(
    pp / "default_full__06_02/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)

nonformal_result = PromptResult(
    pp / 'nonformalised_full__20_04/final',
    colour=Colour("skyblue"),
    full_label="Non-formalised NL prompt",
    short_label="nonformal",
    baseline=gsm_result.mres,
    **result_kwargs
)

formal_result = PromptResult(
    pp / 'formalised_full__16_04/final',
    colour=Colour("steelblue"),
    full_label="Formalised NL prompt",
    short_label="formal",
    baseline=gsm_result.mres,
    **result_kwargs
)

short_code_result = PromptResult(
    pp / 'code_short_full__27_04/final',
    colour=Colour("mediumpurple").lighten(0.2),
    full_label="Shorter code output prompt",
    short_label="code-short",
    baseline=gsm_result.mres,
    **result_kwargs
)

long_code_result = PromptResult(
    pp / 'code_no_sep_full__12_03/corrected',
    colour=Colour("rebeccapurple"),
    full_label="Longer code output prompt",
    short_label="code-long",
    baseline=gsm_result.mres,
    **result_kwargs
)

full_results = {
    'gsm': gsm_result,
    'nonformal': nonformal_result,
    'formal': formal_result,
    'code-short': short_code_result,
    'code-long': long_code_result
}


### Modelling question difficulty

In [ ]:
gsm_result.mres.plot_question_difficulty_per_model(
    # title="Leave-one-(model-)out question difficulty",
    save_prefix=OUTPUTS_FOLDER,
)

In [ ]:
fig_qd_hist = gsm_result.mres.plot_question_difficulty_histogram(
    n_levels=5,
    color=gsm_result.colour.value,
    cumulative_color=gsm_result.colour.darken(factor=0.6).value,
    save_prefix=OUTPUTS_FOLDER,
)
display(fig_qd_hist)

In [ ]:
gsm_result.mres.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

### 1A: Pilot run
Checking on a subset of the benchmark first. Projecting results to the full dataset with alpha = 20%.

50/100 questions, 20/50 template variations -> 1000/5000 total questions in the 'main variant' (20% of the benchmark) (+ 50 questions in the original GSM8K variant)

In [ ]:
pilot_gsm_result.plot_variant_effect(projected_alpha=PROJECTED_ALPHA)

#### Candidate models
Models which, based on the pilot results, are worth checking on the full dataset.

In [ ]:
candidate_models = pilot_gsm_result.get_significant_models(alpha=PROJECTED_ALPHA_THRESHOLD)
candidate_models

In [ ]:
gsm_result.models = candidate_models

### 1B: Full benchmark
Re-running the tests on the full benchmark (100 + 5000 questions) with the identified candidate models.

In [ ]:
gsm_result.plot_variant_effect()

In [ ]:
significant_models = gsm_result.get_significant_models(alpha=ALPHA_THRESHOLD)
significant_models

In [ ]:
for res in (nonformal_result, formal_result, short_code_result, long_code_result):
    res.models = significant_models


## Question 2
Do alternative prompt formats remove the variant dependency?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with code prompt and formalised natural language prompt.

In [ ]:
nonformal_result.plot_variant_effect()

In [ ]:
formal_result.plot_variant_effect()

In [ ]:
short_code_result.plot_variant_effect()

In [ ]:
long_code_result.plot_variant_effect()

### Alternative prompts evaluation
Do alternative prompt formats result in significant performance changes?

Evaluating performance on 'main' variant with the alternative prompts vs GSM-Symbolic prompt.

In [ ]:
nonformal_result.plot_prompt_effect()

In [ ]:
formal_result.plot_prompt_effect()

In [ ]:
short_code_result.plot_prompt_effect()

In [ ]:
long_code_result.plot_prompt_effect()

In [ ]:
long_code_result.plot_acc_change_dist()

In [ ]:
long_code_result.summary()

### GSM8K vs main mean accuracy by model


In [ ]:
all_prompts_summary = pd.concat([r.summary() for r in full_results.values()], keys=[r.short_label for r in full_results.values()], names=['prompt', 'quantity'])
all_prompts_summary

In [ ]:
fig = plot_prompt_acc_evolution(all_prompts_summary, colours={r.short_label: r.colour.value for r in full_results.values()}, models=significant_models, save_prefix=OUTPUTS_FOLDER)
display(fig)

In [ ]:

fig = plot_prompt_comparison(all_prompts_summary, colours={r.short_label: r.colour.lighten(factor=0.3).value for r in full_results.values()}, models=significant_models, save_prefix=OUTPUTS_FOLDER)
display(fig)